In [ ]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict

load_dotenv()

llm = ChatGroq(model = "llama-3.3-70b-versatile")

In [ ]:
class BlogState(TypedDict):
    title: str
    outline: str
    content : str

In [ ]:
def create_outline(state:BlogState) -> BlogState:
    title = state['title']
    prompt = f"Generate an outline for a blog post with the title - {title}"
    response = llm.invoke(prompt)
    state['outline'] = response.content
    return state

def create_blog_content(state:BlogState) -> BlogState:
    outline = state['outline']
    prompt = f"Generate a blog post based on the following outline - {outline}"
    response = llm.invoke(prompt)
    state['content'] = response.content
    return state

In [ ]:
graph = StateGraph(BlogState)

#nodes
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog_content',create_blog_content)

# edges
graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog_content')
graph.add_edge('create_blog_content',END)

#compile
workflow = graph.compile()

In [ ]:
initial_state = BlogState({'title': 'Rise of AI in Defence'})
final_state = workflow.invoke(initial_state)
print(final_state)